# recs_023 -- Stage 3 Ablation B qualitative check (franchise/marketing over-indexing)

Manual read of `query_plus_desc` vs `raw_query` results, checking for the franchise/marketing
over-indexing failure mode flagged in `_scrap/rag_extension.md`.

# Executive Summary

- **Question:** Does appending the query game's description to the query cause retrieval to
  over-index on franchise/studio similarity instead of thematic similarity?
- **Result:** Predicted bias didn't appear -- across 5 max-divergent, franchise/company-tagged
  examples, no result repeated the query's own franchise/company ID. But `query_plus_desc` isn't
  a clean genre win either: 3/5 clear improvements (Terraria, PUBG, Rust), 1/5 regression
  (NieR:Automata -> strategy games instead of action-RPG peers), 1/5 mixed (GTA V).
- **Decision:** Keep `query_plus_desc` as the Ablation B default -- bias absent, quantitative
  edge holds (Hit@K 0.475 vs 0.425). Treat "description helps" as example-dependent.

# Business Context

`query_plus_desc` won on aggregate Hit@K/Recall@K, but the origin plan flagged a risk: IGDB
descriptions are often genre-boilerplate/marketing copy, so appending them could pull retrieval
toward "same franchise/studio" matches instead of genuine thematic similarity -- indistinguishable
from a real win in aggregate metrics. Needs a manual read to tell the two apart.

# Research Question

On the most-divergent top-10 examples between `raw_query` and `query_plus_desc`, do
`query_plus_desc`'s results look like franchise/marketing matches, or genuine thematic matches?

# Hypothesis

`query_plus_desc` results will repeat the query game's own franchise/studio more often than
`raw_query`'s -- the predicted over-indexing failure mode.

**Result: not confirmed.** No franchise/company repeats in either arm. `query_plus_desc` shows
tighter genre alignment in most (3/5) but not all examples -- NieR:Automata is a counter-case.

# Definitions

| Term | Definition | Notes |
|------|------------|-------|
| `overlap@10` | `\|top10(raw_query) ∩ top10(query_plus_desc)\|` for one example | 0 = fully divergent, 10 = identical |
| Ablation B | raw review text vs. review text + query game's own IGDB description, as the query text | `docs/plans/rag_extension_plan.md` decision #1 |
| franchise/marketing over-indexing | retrieval driven by shared IGDB `franchises`/`involved_companies` tags rather than thematic similarity | the failure mode this notebook checks for |

# Data Sources

| Source | Role | Limitation |
|---|---|---|
| `eval_offline_examples.jsonl` (`rag_v1` run) | Full val cohort, 12,500 examples x 6 methods -- uses the 2 `rag_chunk_v1_*` rows per example | No query text stored (Stage 4 gap) -- reads candidate metadata, not review text |
| `igdb_games__enriched.parquet` | Franchise/company/genre lookup by `app_id` | IDs, not names -- read as presence/repeat, not resolved to studio names |

# Design / Process

Manual qualitative read of the already-run `rag_v1` output, not a new eval job.

1. Load `eval_offline_examples.jsonl`, keep the 2 `rag_chunk_v1_*` rows per example.
2. Compute top-10 overlap between arms for all 12,500 examples.
3. Filter to query games with `franchises`/`involved_companies` set in IGDB.
4. Take the 5 most-divergent; print each arm's top-5 with title/franchise/company/genre.
5. Read for pattern: franchise/company repeats vs. genre-level matches.

# Evaluation Outputs / Artifacts

| Artifact | Location | Description |
|---|---|---|
| `eval_retrieval_overall.csv` | `artifacts/recs/offline_eval/runs/rag_v1/` | Quantitative baseline this qualitative check sits on top of |
| `eval_retrieval_by_slice.csv` | `artifacts/recs/offline_eval/runs/rag_v1/` | Slice A/B breakdown referenced in Executive Summary |
| This notebook | `notebooks/retrieval/recs_023_stage3_qualitative_check.ipynb` | Qualitative read; no new artifact produced |

# Notebook Roadmap

1. Quantitative recap (`eval_retrieval_overall.csv`, by-slice)
2. Load Ablation-B examples, compute top-10 overlap per example
3. Select divergent examples with franchise/company-tagged query games
4. Side-by-side comparison, enriched with titles/franchise/company/genre
5. Key findings

# Analysis

## Setup

In [7]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())
RUN_DIR = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v1"
RAW_QUERY = "rag_chunk_v1_raw_query"
QUERY_PLUS_DESC = "rag_chunk_v1_query_plus_desc"

pd.options.display.max_colwidth = 60

## Quantitative Recap

In [8]:
overall = pd.read_csv(RUN_DIR / "eval_retrieval_overall.csv")
by_slice = pd.read_csv(RUN_DIR / "eval_retrieval_by_slice.csv")

methods = [RAW_QUERY, QUERY_PLUS_DESC, "raw", "two_tower_v1"]
print("Overall:")
display(overall.set_index("method").loc[methods, ["Hit@K", "Precision@K", "Recall@K"]])

print("\nBy slice (primary metric: Recall@K for Slice A, Hit@K for Slice B):")
for slice_name in ["slice_a_multi_target", "slice_b_single_target"]:
    sub = by_slice[by_slice["slice_name"] == slice_name].set_index("method").loc[methods, ["Hit@K", "Recall@K"]]
    print(f"\n{slice_name}:")
    display(sub)

Overall:


,Hit@K,Precision@K,Recall@K
method,,,
rag_chunk_v1_raw_query,0.42480,0.004474,0.405454
rag_chunk_v1_query_plus_desc,0.47480,0.005036,0.456046
raw,0.45296,0.004801,0.434516
two_tower_v1,0.51224,0.005414,0.494053



By slice (primary metric: Recall@K for Slice A, Hit@K for Slice B):

slice_a_multi_target:


,Hit@K,Recall@K
method,,
rag_chunk_v1_raw_query,0.722759,0.389207
rag_chunk_v1_query_plus_desc,0.776552,0.453210
raw,0.747586,0.429578
two_tower_v1,0.773793,0.460228



slice_b_single_target:


,Hit@K,Recall@K
method,,
rag_chunk_v1_raw_query,0.406454,0.406454
rag_chunk_v1_query_plus_desc,0.456221,0.456221
raw,0.434820,0.434820
two_tower_v1,0.496136,0.496136


## Load Ablation-B Examples

In [9]:
by_ex: dict[int, dict[str, dict]] = {}
with open(RUN_DIR / "eval_offline_examples.jsonl") as f:
    for line in f:
        d = json.loads(line)
        if d["method"] in (RAW_QUERY, QUERY_PLUS_DESC):
            by_ex.setdefault(d["ex_idx"], {})[d["method"]] = d

n_both = sum(1 for m in by_ex.values() if RAW_QUERY in m and QUERY_PLUS_DESC in m)
print(f"examples with both methods: {n_both}")

examples with both methods: 12500


## Compute Top-10 Overlap

In [10]:
rows = []
for ex_idx, methods_d in by_ex.items():
    if RAW_QUERY not in methods_d or QUERY_PLUS_DESC not in methods_d:
        continue
    raw_top10 = json.loads(methods_d[RAW_QUERY]["retrieved_app_ids_json"])[:10]
    desc_top10 = json.loads(methods_d[QUERY_PLUS_DESC]["retrieved_app_ids_json"])[:10]
    overlap = len(set(raw_top10) & set(desc_top10))
    rows.append(
        dict(
            ex_idx=ex_idx,
            query_app_id=methods_d[RAW_QUERY]["query_app_id"],
            overlap=overlap,
            raw_top10=raw_top10,
            desc_top10=desc_top10,
        )
    )

overlap_df = pd.DataFrame(rows).sort_values("overlap")
print("overlap@10 distribution (0=fully divergent, 10=identical):")
print(overlap_df["overlap"].value_counts().sort_index())

overlap@10 distribution (0=fully divergent, 10=identical):
overlap
0     4801
1     1998
2     1305
3     1073
4      875
5      786
6      605
7      499
8      323
9      182
10      53
Name: count, dtype: int64


## Select Divergent, Franchise-Tagged Examples

In [11]:
igdb = pd.read_parquet(REPO_ROOT / "artifacts" / "igdb" / "igdb_games__enriched.parquet").set_index("app_id")


def field(app_id: int, col: str):
    if app_id not in igdb.index:
        return None
    v = igdb.loc[app_id][col]
    if isinstance(v, np.ndarray):
        return list(v) if v.size else None
    return None if pd.isna(v) else v


def info(app_id: int) -> dict:
    return dict(
        title=field(app_id, "app_name") or field(app_id, "igdb_name") or f"appid:{app_id}",
        franchises=field(app_id, "franchises"),
        companies=field(app_id, "involved_companies"),
        genres=field(app_id, "genres_names"),
    )


# Most-divergent examples where the query game itself carries franchise/company metadata --
# franchise bias, if present, needs this metadata to be checkable.
picked = []
for row in overlap_df.itertuples():
    qinfo = info(row.query_app_id)
    if qinfo["franchises"] or qinfo["companies"]:
        picked.append((row, qinfo))
    if len(picked) >= 5:
        break

print(f"selected {len(picked)} divergent examples with franchise/company-tagged query games")

selected 5 divergent examples with franchise/company-tagged query games


## Side-by-Side Comparison

In [12]:
for row, qinfo in picked:
    print(f"\n{'=' * 90}")
    print(f"ex_idx={row.ex_idx}  query_app_id={row.query_app_id}  overlap@10={row.overlap}")
    print(
        f"QUERY GAME: {qinfo['title']} | franchises={qinfo['franchises']} | "
        f"companies={qinfo['companies']} | genres={qinfo['genres']}"
    )
    print(f"-- {RAW_QUERY} top5 --")
    for a in row.raw_top10[:5]:
        i = info(a)
        print(f"   {a:>8}  {i['title']:<38} franchises={i['franchises']} companies={i['companies']} genres={i['genres']}")
    print(f"-- {QUERY_PLUS_DESC} top5 --")
    for a in row.desc_top10[:5]:
        i = info(a)
        print(f"   {a:>8}  {i['title']:<38} franchises={i['franchises']} companies={i['companies']} genres={i['genres']}")


ex_idx=12491  query_app_id=105600  overlap@10=0
QUERY GAME: Terraria | franchises=None | companies=[np.int64(16887), np.int64(189039), np.int64(188989), np.int64(189041), np.int64(289063), np.int64(289064), np.int64(289065)] | genres=['Platform', 'Role-playing (RPG)', 'Simulator', 'Strategy', 'Adventure', 'Indie']
-- rag_chunk_v1_raw_query top5 --
     823130  Totally Accurate Battlegrounds         franchises=None companies=[np.int64(65368)] genres=['Shooter', 'Indie']
    1118200  People Playground                      franchises=None companies=[np.int64(115528), np.int64(115529)] genres=['Simulator', 'Indie']
    1089980  The Henry Stickmin Collection          franchises=None companies=[np.int64(105468), np.int64(105469)] genres=['Point-and-click', 'Adventure', 'Indie']
     688130  Pogostuck: Rage With Your Friends      franchises=None companies=None genres=['Platform', 'Adventure', 'Indie']
     788260  Rules Of Survival                      franchises=None companies=[np.int64(293

# Key Findings

- **No franchise/marketing over-indexing.** Across the 5 most-divergent, franchise/company-tagged
  examples (Terraria, PUBG, Rust, NieR:Automata, GTA V), zero results in either arm repeated the
  query game's own franchise/company ID.
- **`query_plus_desc` genre-coherence: 3/5 wins, 1/5 regression, 1/5 mixed.** Terraria/PUBG/Rust:
  `query_plus_desc` lands on genre-mates (sandbox-survival, survival-crafting) where `raw_query`
  lands on unrelated games. NieR:Automata inverts: `raw_query`'s action-RPG peers (Devil May Cry,
  Toukiden 2) are tighter than `query_plus_desc`'s drift into strategy games (Stellaris, XCOM 2).
  GTA V: one strong hit (Saints Row III), two weak ones.
- **Net:** matches the aggregate Hit@K edge, but not a clean sweep -- treat as example-dependent,
  not universal.

# Recommendation / Next Steps

**Action:** Keep `query_plus_desc` as the Ablation B default -- predicted bias absent,
quantitative edge holds. Both RAG arms still trail `two_tower_v1` on primary metrics, so this is
a research finding, not a promotion decision.

**Risk:** n=5, hand-picked for max divergence -- not random/exhaustive. NieR:Automata shows the
description can push toward a different-but-still-legitimate genre neighborhood that's a worse
match -- subtler than the franchise-bias hypothesis tested here.

**Follow-ups:** larger random sample if this arm ships; automate a franchise-repeat-rate metric
and a genre-overlap metric across the full cohort instead of manual read.

**Open:** does this hold at k=100, not just top-5 shown here; does the `two_tower_v1` gap close
with embedder swap / blend-weight tuning (see `recs_024`/`recs_025`).